Création de la base de données:
on commence par récupérer les données financières sur les taux de change et les futures sur les commodities, puis on réxupère les données météo grâce à l'API open meteo.

In [1]:
!pip install yfinance

In [2]:
!git clone https://github.com/schampon/skfin.git
%cd skfin
!pip install -e .

Cloning into 'skfin'...
remote: Enumerating objects: 677, done.
remote: Counting objects: 100% (129/129), done.
remote: Compressing objects: 100% (101/101), done.
remote: Total 677 (delta 40), reused 83 (delta 27), pack-reused 548 (from 2)
Receiving objects: 100% (677/677), 142.04 MiB | 17.63 MiB/s, done.
Resolving deltas: 100% (298/298), done.
/Users/pierrepalacin/Desktop/M2/S1/ENSAE/ML for trading/Project/Project/Machine-Learning-for-Portfolio-Management/skfin
Obtaining file:///Users/pierrepalacin/Desktop/M2/S1/ENSAE/ML%20for%20trading/Project/Project/Machine-Learning-for-Portfolio-Management/skfin
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
INFO: pip is looking at multiple versions of skfin to determine which version is compatible with other requirements. This could take a while.
ERROR: Package 'skfin' requires a differen

In [3]:
!pip install openmeteo-requests
!pip install requests-cache retry-requests numpy pandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 873.8/873.8 kB 12.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 647.5/647.5 kB 17.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.7/4.7 MB 14.9 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8/8 [openmeteo-requests]rllib3-future]
  Using cached attrs-26.1.0-py3-none-any.whl.metadata (8.8 kB)
Using cached attrs-26.1.0-py3-none-any.whl (67 kB)
  Attempting uninstall: attrs
    Found existing installation: attrs 21.4.0
    Uninstalling attrs-21.4.0:
      Successfully uninstalled attrs-21.4.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6/6 [requests-cache]
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
spyder 5.3.3 requires pyqt5<5.16, which is not installed.
spyder 5.3.3 requires pyqtwebengine<5.16, which is not installed.


In [4]:
import yfinance as yf
import pandas as pd
import numpy as np
import seaborn as sns
import requests
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output
from datetime import datetime, timedelta
from skfin.estimators import MLPRegressor, MultiLGBMRegressor, Ridge
from skfin.backtesting import Backtester
from skfin.estimators import MeanVariance, Ridge
from skfin.datasets import load_kf_returns
from skfin.plot import line
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.0.2 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "/Users/pierrepalacin/opt/anaconda3/lib/python3.9/runpy.py", line 197, in _run_module_as_main
    return _run_code(code, main_globals, None,
  File "/Users/pierrepalacin/opt/anaconda3/lib/python3.9/runpy.py", line 87, in _run_code
    exec(code, run_globals)
  File "/Users/pierrepalacin/opt/anaconda3/lib/python3.9/site-packages/ipykernel_launcher.py", line 17, in <module>
    app.launch_new_instance()
  File "/Users/pierrepalacin/opt/anaconda3/lib/python3.9/site-packages/traitlets/config/application.py", line 846, i

AttributeError: _ARRAY_API not found


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.0.2 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "/Users/pierrepalacin/opt/anaconda3/lib/python3.9/runpy.py", line 197, in _run_module_as_main
    return _run_code(code, main_globals, None,
  File "/Users/pierrepalacin/opt/anaconda3/lib/python3.9/runpy.py", line 87, in _run_code
    exec(code, run_globals)
  File "/Users/pierrepalacin/opt/anaconda3/lib/python3.9/site-packages/ipykernel_launcher.py", line 17, in <module>
    app.launch_new_instance()
  File "/Users/pierrepalacin/opt/anaconda3/lib/python3.9/site-packages/traitlets/config/application.py", line 846, i

AttributeError: _ARRAY_API not found

/Users/pierrepalacin/opt/anaconda3/lib/python3.9/site-packages/scipy/__init__.py:155: UserWarning: A NumPy version >=1.18.5 and <1.25.0 is required for this version of SciPy (detected version 2.0.2
  warnings.warn(f"A NumPy version >={np_minversion} and <{np_maxversion}"


ValueError: numpy.dtype size changed, may indicate binary incompatibility. Expected 96 from C header, got 88 from PyObject

**Première sous partie** : Récupération des données financières. On veut associer un pays à une matière première, on construit donc un dictionnaire pour associer la devise à la MP.


*   L'Australie est considérée comme un exportateur de gaz naturel.  
*   Le Brésil, la Malaisie, la Norvège la Russie et l'Afrique du Sud sont les exportateurs de pétrole
*   trop bizarre : CL=F représente le crude oil

On commence par définir les paramètres pour l'API : la durée et les pays exportateurs de matières premières en mettant dans ce dictionnaire les tickers des taux de change ainsi que ceux des futures.



In [ ]:
period = "20y"

markets = {"Australia": {"fx": "AUD=X","commodity": "NG=F"},
    "Brazil": {"fx": "BRL=X","commodity": "BZ=F"},
    "Canada": {"fx": "CAD=X","commodity": "CL=F"},
    "Malaysia": {"fx": "MYR=X","commodity": "BZ=F"},
    "Mexico": {"fx": "MXN=X","commodity": "CL=F"},
    "Norway": {"fx": "NOK=X","commodity": "BZ=F"},
    "Russia": {"fx": "RUB=X","commodity": "BZ=F"},
    "South_Africa": {"fx": "ZAR=X","commodity": "BZ=F"}} #dictionnaire de dictionnaires?

#qu'est ce que l'on fait aec ce set?
tickers = sorted(set([v["fx"] for v in markets.values()] +[v["commodity"] for v in markets.values()])) #ici, on utilise un set pour stocker les deux types de tickers, à étayer.

df_raw = yf.download(tickers,period=period,progress=False)["Close"]

df_raw = df_raw.sort_index()

/tmp/ipykernel_1211/3754420613.py:15: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df_raw = yf.download(tickers,period=period,progress=False)["Close"]


In [ ]:
display(df_raw)

Ticker,AUD=X,BRL=X,BZ=F,CAD=X,CL=F,MXN=X,MYR=X,NG=F,NOK=X,RUB=X,ZAR=X
Date,,,,,,,,,,,
2006-10-23,1.321000,2.134300,NaN,1.12840,NaN,10.840000,3.6739,NaN,6.69960,26.916000,7.680100
2006-10-24,1.317000,2.141000,NaN,1.12660,NaN,10.810000,3.6764,NaN,6.66460,26.903000,7.705000
2006-10-25,1.312900,2.144300,NaN,1.12400,61.400002,10.734000,3.6681,7.693,6.58330,26.853001,7.553700
2006-10-26,1.308000,2.131500,NaN,1.12270,60.360001,10.715500,3.6600,7.497,6.53170,26.768000,7.483000
2006-10-27,1.300100,2.131500,NaN,1.11950,60.750000,10.658000,3.6446,7.153,6.54130,26.724001,7.407600
...,...,...,...,...,...,...,...,...,...,...,...
2026-10-20,1.432090,5.037000,108.790001,1.41053,101.790001,18.409639,4.0800,2.963,9.42195,82.743088,16.418631
2026-10-21,1.427230,5.022700,108.820000,1.40662,97.150002,18.424431,4.0797,2.790,9.40251,82.781227,16.390690
2026-10-22,1.422617,5.020391,113.169998,1.40420,99.470001,18.678699,4.0787,2.796,9.36173,82.097870,16.370050


In [ ]:
market_data = {} 

for country, info in markets.items(): 

    temp = pd.DataFrame(index=df_raw.index) 

    temp["FX"] = df_raw[info["fx"]]
    temp["Commodity"] = df_raw[info["commodity"]]

    market_data[country] = temp.dropna()


def rolling_zscore(x, window=60):
    mean = x.rolling(window).mean()
    std = x.rolling(window).std()
    return (x - mean) / std


for country, df in market_data.items():

    df["log_commodity"] = np.log(df["Commodity"])
    df["log_fx"] = np.log(df["FX"])

    df["commodity_z"] = rolling_zscore(df["log_commodity"], window=60) 
    df["fx_z"] = rolling_zscore(df["log_fx"], window=60)

    df["spread"] = df["commodity_z"] - df["fx_z"]
#C'est ce feature qui nous permettra d'exploiter la lead lag convergence.

horizon = 15

for country, df in market_data.items():
    df["target"] = (df["spread"].shift(-horizon)- df["spread"]) 

/usr/local/lib/python3.13/dist-packages/pandas/core/arraylike.py:399: RuntimeWarning: invalid value encountered in log
  result = getattr(ufunc, method)(*inputs, **kwargs)
/usr/local/lib/python3.13/dist-packages/pandas/core/arraylike.py:399: RuntimeWarning: invalid value encountered in log
  result = getattr(ufunc, method)(*inputs, **kwargs)
/tmp/ipykernel_1211/2999630379.py:21: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df["log_commodity"] = np.log(df["Commodity"])
/tmp/ipykernel_1211/2999630379.py:22: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/i

In [ ]:
display(market_data) 

{'Australia':                   FX  Commodity  log_commodity    log_fx  commodity_z  \
 Date                                                                    
 2006-10-25  1.312900      7.693       2.040311  0.272238          NaN   
 2006-10-26  1.308000      7.497       2.014503  0.268499          NaN   
 2006-10-27  1.300100      7.153       1.967532  0.262441          NaN   
 2006-10-30  1.300100      7.416       2.003640  0.262441          NaN   
 2006-10-31  1.290300      7.534       2.019426  0.254875          NaN   
 ...              ...        ...            ...       ...          ...   
 2026-10-19  1.434288      3.058       1.117761  0.360669     0.857308   
 2026-10-20  1.432090      2.963       1.086202  0.359135     0.306641   
 2026-10-21  1.427230      2.790       1.026042  0.355736    -0.739240   
 2026-10-22  1.422617      2.796       1.028190  0.352498    -0.714595   
 2026-10-23  1.424542      3.050       1.115142  0.353850     0.767966   
 
                 fx_z  

**Données environnementales** : dans cette sous partie, on utilise l'api open meteo pour récupérer les données météorologiques des pays exportateurs de commodities.

In [ ]:
EMDAT_PATH = "/content/public_emdat_2026-10-19.xlsx"
emdat = pd.read_excel(EMDAT_PATH)

required_columns = ["Disaster Group", "Disaster Subgroup", "Disaster Type",
                    "Country", "Start Year", "Start Month", "Start Day"]

missing_columns = [col for col in required_columns if col not in emdat.columns]
if missing_columns:
    raise ValueError(f"Colonnes EM-DAT manquantes : {missing_columns}")

emdat_natural = emdat.loc[emdat["Disaster Group"] == "Natural"].copy()

emdat_natural["event_date"] = pd.to_datetime(
    dict(year=emdat_natural["Start Year"],
         month=emdat_natural["Start Month"],
         day=emdat_natural["Start Day"]),
    errors="coerce"
)
emdat_natural = emdat_natural.dropna(subset=["event_date"])

country_emdat = {
    "Australia": "Australia",
    "Brazil": "Brazil",
    "Canada": "Canada",
    "Malaysia": "Malaysia",
    "Mexico": "Mexico",
    "Norway": "Norway",
    "Russia": "Russian Federation",
    "South_Africa": "South Africa"}

disaster_data = {}

for country in markets:
    trading_index = market_data[country].index
    events = emdat_natural.loc[emdat_natural["Country"] == country_emdat[country]].copy()

    mapped_dates = [] #si la catastrophe se produit un weekend on la rattache au lundi
    for event_date in events["event_date"]:
        position = trading_index.searchsorted(event_date)
        mapped_dates.append(trading_index[position] if position < len(trading_index) else pd.NaT)
    events["market_date"] = mapped_dates
    events = events.dropna(subset=["market_date"])

    daily = pd.DataFrame(index=trading_index)

    # Indicatrice du nombre de catastrophes
    start_count = events.groupby("market_date").size()
    daily["disaster_start_count"] = start_count.reindex(trading_index, fill_value=0)
    daily["disaster_start"] = (daily["disaster_start_count"] > 0).astype(int)

    # classification par type de catastrophe.
    for subgroup in sorted(events["Disaster Subgroup"].dropna().unique()):
        subgroup_name = subgroup.lower().replace(" ", "_").replace("-", "_")
        subgroup_events = events.loc[events["Disaster Subgroup"] == subgroup]
        subgroup_count = subgroup_events.groupby("market_date").size()
        daily[f"start_{subgroup_name}"] = subgroup_count.reindex(trading_index, fill_value=0)

    # Mémoire récente des catastrophes : uniquement à partir des événements déjà survenus.
    for window in [5, 15, 30]:
        daily[f"disaster_starts_{window}d"] = (
            daily["disaster_start_count"].rolling(window=window, min_periods=1).sum()
        )

    last_event = pd.Series(pd.NaT, index=trading_index, dtype="datetime64[ns]")
    event_days = daily.index[daily["disaster_start"] == 1]
    last_event.loc[event_days] = event_days
    last_event = last_event.ffill()
    daily["days_since_disaster"] = (daily.index.to_series() - last_event).dt.days

    disaster_data[country] = daily


**Troisième sous partie** : fusion des bases de données financières et environnementales.

In [ ]:
for country in markets:
    new_cols = disaster_data[country].columns.difference(market_data[country].columns)
    if not new_cols.empty:
        market_data[country] = market_data[country].join(disaster_data[country][new_cols], how="left")

display(market_data["Brazil"].head())

,FX,Commodity,log_commodity,log_fx,commodity_z,fx_z,spread,target,disaster_start_count,disaster_start,start_biological,start_climatological,start_geophysical,start_hydrological,start_meteorological,disaster_starts_5d,disaster_starts_15d,disaster_starts_30d,days_since_disaster
Date,,,,,,,,,,,,,,,,,,,
2007-07-30,1.8836,75.739998,4.327306,0.633185,NaN,NaN,NaN,NaN,28,1,1,0,0,24,3,28.0,28.0,28.0,0
2007-07-31,1.8655,77.050003,4.344455,0.623529,NaN,NaN,NaN,NaN,0,0,0,0,0,0,0,28.0,28.0,28.0,1
2007-08-01,1.8779,75.349998,4.322144,0.630154,NaN,NaN,NaN,NaN,0,0,0,0,0,0,0,28.0,28.0,28.0,2
2007-08-02,1.8659,75.760002,4.327570,0.623744,NaN,NaN,NaN,NaN,0,0,0,0,0,0,0,28.0,28.0,28.0,3
2007-08-03,1.8750,74.750000,4.314149,0.628609,NaN,NaN,NaN,NaN,0,0,0,0,0,0,0,28.0,28.0,28.0,4
